# Replication completion timing bounds

This notebook maps Repli-seq data to replication timing profiles, fits initiation-rate landscapes, runs stochastic replication simulations, and compares completion-time statistics with the analytical bounds developed by Alkhaled et al. (2026).

The analysis is split into two ready-to-run workflows.

1. **Non-periodic chromosome profiles.** These are compared with the full-line bound $\mathbb{R}$.
2. **Periodic interval profiles.** These are compared with the torus bound $\mathbb{T}_L$.

Repli-seq extraction tools and initiation-rate fitting follow [Berkemeier et al. (2025)](https://www.nature.com/articles/s41467-025-59991-w), implemented in `replication_src.py`.

## Imports and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from replication_src import *
from replication_data_sources import *

# Analysis A: full-line chromosome profiles

This section analyses non-periodic chromosome-scale timing profiles. The fit and simulation use `perQ=False`, and the theoretical comparison uses the full-line completion bound. Because chromosome-wide simulations can be large, the default run is organised one profile at a time.


## A1. Load UCSC Repli-seq signal

Choose ENCODE/UW Repli-seq cell lines, download the UCSC wavelet-smoothed timing BigWigs, then load locally available tracks as 1 kb NumPy arrays. UCSC's `WaveSignal` track has already collapsed the cell-cycle fractions into one weighted timing profile, so this step does not reconstruct timing from S-phase bins. The raw UCSC signal is cached here. In A2, user-defined centromere intervals in `REPLISEQ_CENTROMERES_BP` are applied during timing preparation, after smoothing but before the final `timing_range` rescale.

In [ ]:
UCSC_REPLISEQ_BASE_URL = "http://hgdownload.soe.ucsc.edu/goldenPath/hg19/encodeDCC/wgEncodeUwRepliSeq"
REPLISEQ_REPLICATE = 1
REPLISEQ_CHROMS = ["chr1"]
REPLISEQ_RESOLUTION = 1_000
REPLISEQ_OVERWRITE_BIGWIG = False
REPLISEQ_OVERWRITE_TIMING_CACHE = False

UCSC_REPLISEQ_CELL_LINE_NAMES = {
    "BG02ES": "Bg02es",
    "BJ": "Bj",
    "HELAS3": "Helas3",
    "HEPG2": "Hepg2",
    "HUVEC": "Huvec",
    "IMR90": "Imr90",
    "K562": "K562",
    "MCF7": "Mcf7",
    "NHEK": "Nhek",
    "SKNSH": "Sknsh",
}
REPLISEQ_CELL_LINES = list(UCSC_REPLISEQ_CELL_LINE_NAMES)


In [ ]:
REPLISEQ_BIGWIGS = download_ucsc_repliseq_wavelet_bigwigs(
    cell_lines=REPLISEQ_CELL_LINES,
    base_url=UCSC_REPLISEQ_BASE_URL,
    replicate=REPLISEQ_REPLICATE,
    data_dir=DATA_DIR,
    overwrite=REPLISEQ_OVERWRITE_BIGWIG,
    cell_line_names=UCSC_REPLISEQ_CELL_LINE_NAMES,
)

display(ucsc_repliseq_bigwig_status_table(
    REPLISEQ_BIGWIGS,
    replicate=REPLISEQ_REPLICATE,
))

In [ ]:
LINE_PROFILE_DATASETS, REPLISEQ_WAVELET_ARRAYS, line_timing_summary = build_ucsc_repliseq_line_datasets(
    cell_lines=REPLISEQ_CELL_LINES,
    chroms=REPLISEQ_CHROMS,
    resolution=REPLISEQ_RESOLUTION,
    replicate=REPLISEQ_REPLICATE,
    data_dir=DATA_DIR,
    timing_dir=TIMING_DIR,
    overwrite_timing_cache=REPLISEQ_OVERWRITE_TIMING_CACHE,
    cell_line_names=UCSC_REPLISEQ_CELL_LINE_NAMES,
)
LINE_KEY = next(iter(LINE_PROFILE_DATASETS))
repliseq_wavelet_signal = REPLISEQ_WAVELET_ARRAYS[LINE_KEY]["signal"]
repliseq_wavelet_positions = REPLISEQ_WAVELET_ARRAYS[LINE_KEY]["positions"]
repliseq_wavelet_bigwig = REPLISEQ_WAVELET_ARRAYS[LINE_KEY]["bigwig"]

display(line_timing_summary)
display(timing_cache_table(LINE_PROFILE_DATASETS))

## A2. Run the UCSC Repli-seq full-line analysis

Run the full-line analysis from one UCSC wavelet-signal CSV written in A1. Set `LINE_ANALYSIS_CELL_LINE` and `LINE_ANALYSIS_CHROM` in the code cell below to choose the dataset. `run_single_dataset` applies the general `REPLISEQ_CENTROMERES_BP` interval(s) during timing preparation: after interpolation/refinement/smoothing, it sets the selected bins before rescaling to the pre-rescale value that maps to the minimum final replication time, and only then rescales to `timing_range`. When the first `timing_range` endpoint is larger than the second, this uses the maximum pre-rescale signal so the centromere bins become the lower final endpoint after rescaling. The same centromere bins are then ignored in both empirical and theoretical `T_epsilon` and expected-time summaries. The signal is already on a 1 kb grid; `refine_factor=1` keeps that grid, while smaller values downsample it, for example `refine_factor=.1` gives a 10 kb grid. The simulation is non-periodic: `perQ=False`.

The speed is entered below in physical units, `fork_speed_kb_min=1.4`. `run_single_dataset` converts it internally using the actual grid spacing and prints the converted value. With a 1 kb grid this becomes `1.4` grid sites/min. Do not manually divide the speed before passing it here.


In [ ]:
LINE_ANALYSIS_CELL_LINE = "SKNSH"
LINE_ANALYSIS_CHROM = "chr1"
REPLISEQ_CENTROMERES_BP = [(115_000_000, 145_000_000)]

LINE_ANALYSIS_CELL_LINE_KEY = str(LINE_ANALYSIS_CELL_LINE).upper().replace("-", "").replace("_", "").replace(" ", "")
LINE_KEY = f"{LINE_ANALYSIS_CELL_LINE_KEY}_line_{safe_filename(LINE_ANALYSIS_CHROM)}"

if LINE_KEY not in LINE_PROFILE_DATASETS:
    available = ", ".join(LINE_PROFILE_DATASETS)
    raise KeyError(f"{LINE_KEY} is not available. Available keys: {available}")

line_selected_cfg = LINE_PROFILE_DATASETS[LINE_KEY]
display(pd.DataFrame([{
    "key": LINE_KEY,
    "cell_line": line_selected_cfg["cell_line"],
    "chromosome": line_selected_cfg["chrom"],
    "resolution_bp": line_selected_cfg["resolution"],
    "source_bigwig": line_selected_cfg["source_bigwig"],
}]))

LINE_ANALYSIS_SIM_NUMBER = 1000
LINE_ANALYSIS_REFINE_FACTOR = .1
LINE_ANALYSIS_SMOOTH_WINDOW = 100
LINE_ANALYSIS_TIMING_RANGE = (550, 30)
LINE_ANALYSIS_EPS_GRID = np.geomspace(1e-2, 0.99, 100)
LINE_ANALYSIS_NUM_T_BOUND = 4_000
LINE_ANALYSIS_MAX_REP_TIME = 2_000

line_result = run_single_dataset(
    line_selected_cfg,
    fork_speed_kb_min=1.4,
    sim_number=LINE_ANALYSIS_SIM_NUMBER,
    refine_factor=LINE_ANALYSIS_REFINE_FACTOR,
    smooth_window=LINE_ANALYSIS_SMOOTH_WINDOW,
    timing_range=LINE_ANALYSIS_TIMING_RANGE,
    eps_grid=LINE_ANALYSIS_EPS_GRID,
    num_t_bound=LINE_ANALYSIS_NUM_T_BOUND,
    max_rep_time=LINE_ANALYSIS_MAX_REP_TIME,
    timing_cache_mode="load",
    centromeres_bp=REPLISEQ_CENTROMERES_BP,
)

## A3. Plot and summarize the selected full-line result

Plot the standard diagnostics for the single full-line result from A2.

In [ ]:
make_standard_plots(
    line_result,
    save_figures=False,
    initiation_ylims=(1e-4, 1e-2),

)

expected_time_summary_table({LINE_KEY: line_result})

## A4. Full-line batch scatter across selected chr1 cell lines

Choose one or more UCSC Repli-seq cell lines, run the full-line chr1 analysis for those tracks, and show the empirical-versus-theoretical expected-time scatter. The batch uses the same A2 timing-preparation settings and centromere handling, and ignores those bins in both empirical and theoretical completion summaries.

In [ ]:
LINE_BATCH_CELL_LINES = REPLISEQ_CELL_LINES

LINE_BATCH_CHROM = "chr1"
LINE_BATCH_SIM_NUMBER = 1000
LINE_BATCH_REFINE_FACTOR = globals().get("LINE_ANALYSIS_REFINE_FACTOR", .1)
LINE_BATCH_SMOOTH_WINDOW = globals().get("LINE_ANALYSIS_SMOOTH_WINDOW", 100)
LINE_BATCH_TIMING_RANGE = globals().get("LINE_ANALYSIS_TIMING_RANGE", (550, 30))
LINE_BATCH_EPS_GRID = globals().get("LINE_ANALYSIS_EPS_GRID", np.geomspace(1e-2, 0.99, 100))
LINE_BATCH_NUM_T_BOUND = globals().get("LINE_ANALYSIS_NUM_T_BOUND", 4_000)
LINE_BATCH_MAX_REP_TIME = globals().get("LINE_ANALYSIS_MAX_REP_TIME", 2_000)

LINE_BATCH_KEYS = [
    f"{str(cell_line).upper().replace('-', '').replace('_', '').replace(' ', '')}_line_{safe_filename(LINE_BATCH_CHROM)}"
    for cell_line in LINE_BATCH_CELL_LINES
]
missing_line_batch_keys = [key for key in LINE_BATCH_KEYS if key not in LINE_PROFILE_DATASETS]
if missing_line_batch_keys:
    available = ", ".join(LINE_PROFILE_DATASETS)
    missing = ", ".join(missing_line_batch_keys)
    raise KeyError(f"Missing batch dataset key(s): {missing}. Available keys: {available}")

LINE_BATCH_DATASETS = {key: LINE_PROFILE_DATASETS[key] for key in LINE_BATCH_KEYS}
display(pd.DataFrame([
    {
        "key": key,
        "cell_line": cfg["cell_line"],
        "chromosome": cfg["chrom"],
        "resolution_bp": cfg["resolution"],
        "source_bigwig": cfg["source_bigwig"],
    }
    for key, cfg in LINE_BATCH_DATASETS.items()
]))

line_batch_results = run_dataset_collection(
    LINE_BATCH_DATASETS,
    selected_keys=LINE_BATCH_KEYS,
    fork_speed_kb_min=1.4,
    sim_number=LINE_BATCH_SIM_NUMBER,
    refine_factor=LINE_BATCH_REFINE_FACTOR,
    smooth_window=LINE_BATCH_SMOOTH_WINDOW,
    timing_range=LINE_BATCH_TIMING_RANGE,
    eps_grid=LINE_BATCH_EPS_GRID,
    num_t_bound=LINE_BATCH_NUM_T_BOUND,
    max_rep_time=LINE_BATCH_MAX_REP_TIME,
    timing_cache_mode="load",
    centromeres_bp=REPLISEQ_CENTROMERES_BP,
)

In [ ]:
fig, ax, line_expected_time_pairs = plot_expected_time_pair_scatter(
    line_batch_results,
    title=f"Full line {LINE_BATCH_CHROM}: expected local replication-time bound across cell lines",
    label_col="cell_line",
    xlims=(0, 800),
    ylims=(0, 800),
)
fig.savefig(FIGURE_DIR / f"line_expected_time_pairs_{safe_filename(LINE_BATCH_CHROM)}.pdf", bbox_inches="tight")
plt.show()

# Analysis B: finite HCT116 chr8 intervals on the torus

For the finite-domain analyses, we use high-resolution HCT116 Repli-seq data from Zhao et al.~\cite{zhao2020high} over three chromosome~8 windows motivated by Jaworski et al.~\cite{jaworski2025ecdna}. These are the 7~Mb c-MYC/ecDNA context window `chr8:124,000,000--131,000,000`, the dominant ecDNA-derived interval `chr8:126,425,747--127,997,820`, and a c-MYC-centred subwindow corresponding to the origin-density zoom in that study, `chr8:127,675,747--127,775,747`. We treat these finite windows as synthetic periodic domains to test the torus estimates on circularised, amplicon-scale initiation landscapes. The simulations should therefore be interpreted as HCT116 Repli-seq-derived timing profiles evaluated on genomic intervals chosen from the chromosome~8 regions highlighted by the ecDNA study, rather than as purified ecDNA Repli-seq measurements.

## B1. Download and load Zhao HCT116 Repli-seq timing

Download the processed HCT116 Repli-seq matrix from GSE137764, reconstruct a timing value from the 16 S-phase signal rows, interpolate it to a 1 kb analysis grid, and cache one raw timing CSV for each selected chr8 finite window.

In [ ]:
ZHAO_HCT116_GSE_ACCESSION = DEFAULT_ZHAO_HCT116_GSE_ACCESSION
ZHAO_HCT116_REPLISEQ_URL = DEFAULT_ZHAO_HCT116_REPLISEQ_URL
ZHAO_HCT116_REPLISEQ_GZ = DATA_DIR / DEFAULT_ZHAO_HCT116_REPLISEQ_FILENAME_GZ
ZHAO_HCT116_REPLISEQ_TABLE = DATA_DIR / DEFAULT_ZHAO_HCT116_REPLISEQ_FILENAME
ZHAO_HCT116_SOURCE_RESOLUTION = DEFAULT_ZHAO_HCT116_SOURCE_RESOLUTION
ZHAO_HCT116_ANALYSIS_RESOLUTION = DEFAULT_ZHAO_HCT116_ANALYSIS_RESOLUTION
ZHAO_HCT116_S_PHASE_BINS = DEFAULT_ZHAO_HCT116_S_PHASE_BINS
ZHAO_HCT116_OVERWRITE_DOWNLOAD = False
ZHAO_HCT116_OVERWRITE_TIMING_CACHE = False

zhao_hct116_table_path = ensure_zhao_hct116_repliseq(
    url=ZHAO_HCT116_REPLISEQ_URL,
    gz_path=ZHAO_HCT116_REPLISEQ_GZ,
    table_path=ZHAO_HCT116_REPLISEQ_TABLE,
    overwrite=ZHAO_HCT116_OVERWRITE_DOWNLOAD,
)
zhao_hct116_matrix = load_zhao_hct116_repliseq_matrix(
    path=zhao_hct116_table_path,
    s_phase_bins=ZHAO_HCT116_S_PHASE_BINS,
    source_resolution=ZHAO_HCT116_SOURCE_RESOLUTION,
)

PERIODIC_CELL_LINES = ["HCT116"]
PERIODIC_REGIONS = [
    {
        "region_id": "CHR8_CMYC_7MB",
        "label": "chr8 c-MYC/ecDNA context window",
        "short_label": "chr8 c-MYC/ecDNA 7 Mb",
        "chrom": "chr8",
        "start": 124_000_000,
        "end": 131_000_000,
    },
    {
        "region_id": "CHR8_ECDNA_DOMINANT",
        "label": "chr8 dominant ecDNA-derived interval",
        "short_label": "chr8 dominant ecDNA interval",
        "chrom": "chr8",
        "start": 126_425_747,
        "end": 127_997_820,
    },
    {
        "region_id": "CHR8_CMYC_ORIGIN_ZOOM",
        "label": "chr8 c-MYC-centred origin-density zoom",
        "short_label": "chr8 c-MYC origin zoom",
        "chrom": "chr8",
        "start": 127_675_747,
        "end": 127_775_747,
    },
]

PERIODIC_INTERVAL_DATASETS = build_zhao_hct116_periodic_configs(
    PERIODIC_REGIONS,
    resolution=ZHAO_HCT116_ANALYSIS_RESOLUTION,
    accession=ZHAO_HCT116_GSE_ACCESSION,
    source_resolution=ZHAO_HCT116_SOURCE_RESOLUTION,
    cell_line="HCT116",
)

display(pd.DataFrame([
    {
        "key": key,
        "cell_line": cfg["cell_line"],
        "region": cfg["short_label"].replace(f"{cfg['cell_line']} ", ""),
        "coordinates": f"{cfg.get('requested_chrom', cfg['chrom'])}:{cfg['start']}-{cfg['end']}",
        "source_resolution_bp": cfg["source_resolution"],
        "analysis_resolution_bp": cfg["resolution"],
    }
    for key, cfg in PERIODIC_INTERVAL_DATASETS.items()
]))

display(timing_cache_table(PERIODIC_INTERVAL_DATASETS))

periodic_timing_preprocessing = preprocess_zhao_hct116_timing_collection(
    PERIODIC_INTERVAL_DATASETS,
    zhao_hct116_matrix,
    overwrite=ZHAO_HCT116_OVERWRITE_TIMING_CACHE,
)
display(periodic_timing_preprocessing)

## B2. Run one HCT116 finite-window analysis

Choose one cached HCT116 finite interval and run it as a synthetic periodic domain. The cached timing arrays are already on the 1 kb analysis grid, so `refine_factor=1` keeps the grid unchanged before smoothing and fitting.

In [ ]:
PERIODIC_EXAMPLE_CELL_LINE = "HCT116"
PERIODIC_EXAMPLE_REGION_ID = "CHR8_CMYC_7MB"
PERIODIC_KEY = f"{PERIODIC_EXAMPLE_CELL_LINE}_{PERIODIC_EXAMPLE_REGION_ID}"

if PERIODIC_KEY not in PERIODIC_INTERVAL_DATASETS:
    available = ", ".join(PERIODIC_INTERVAL_DATASETS)
    raise KeyError(f"{PERIODIC_KEY} is not available. Available keys: {available}")

periodic_result = run_single_dataset(
    PERIODIC_INTERVAL_DATASETS[PERIODIC_KEY],
    fork_speed_kb_min=1.4,
    sim_number=10000,
    refine_factor=1,
    smooth_window=10,
    timing_range=(60, 10),
    eps_grid=np.geomspace(1e-2, 0.99, 100),
    num_t_bound=4_000,
    max_rep_time=2_000,
    timing_cache_mode="load",
)

## B3. Plot and summarize the selected periodic result

Plot the standard diagnostics for the single periodic result from B2.

In [ ]:
make_standard_plots(
    periodic_result,
    save_figures=False,
)

expected_time_summary_table({PERIODIC_KEY: periodic_result})

## B4. HCT116 chr8 finite-window batch scatter

Run the three selected HCT116 chr8 finite windows and show the empirical-versus-theoretical expected-time scatter for the torus bound.

In [ ]:
PERIODIC_BATCH_DATASETS = PERIODIC_INTERVAL_DATASETS

periodic_batch_results = run_dataset_collection(
    PERIODIC_BATCH_DATASETS,
    selected_keys=list(PERIODIC_BATCH_DATASETS),
    fork_speed_kb_min=1.4,
    sim_number=1000,
    refine_factor=1,
    smooth_window=50,
    timing_range=(60, 10),
    eps_grid=np.geomspace(1e-4, 0.99, 100),
    num_t_bound=4_000,
    max_rep_time=2_000,
    timing_cache_mode="load",
)

In [ ]:
fig, ax, periodic_expected_time_pairs = plot_expected_time_pair_scatter(
    periodic_batch_results,
    title="Torus: expected local replication-time bound for HCT116 chr8 finite windows",
    xlims=(0, 100),
    ylims=(0, 100),
)
fig.savefig(FIGURE_DIR / "hct116_chr8_three_windows_torus_expected_time_pairs.pdf", bbox_inches="tight")
plt.show()

## B5. Fixed-epsilon completion-time absolute-difference heatmap

Scan centred HCT116 chr8 windows around 127 Mb. For each `variable_x`, the genomic interval is `chr8:(127000-variable_x) kb` to `chr8:(127000+variable_x) kb`; for each `variable_t`, the timing rescale range is `(30, variable_t)`. Set `B5_PERIODIC` before running the simulations so fitting, simulation, and the theoretical geometry are chosen consistently.


In [ ]:
B5_EPSILON = 0.05
B5_PERIODIC = True
B5_CENTER_KB = 127_000
B5_VARIABLE_X_KB = np.array([50, 100, 250, 500, 786, 1_000, 2_000, 3_500], dtype=float)
B5_VARIABLE_T_MIN = np.array([40, 50, 60, 70, 80], dtype=float)
B5_TIME_RANGE_START_MIN = 30
B5_FORK_SPEED_KB_MIN = 1.4
B5_SIM_NUMBER = 500
B5_REFINE_FACTOR = 1
B5_SMOOTH_WINDOW = 10
B5_NUM_T_BOUND = 4_000
B5_MAX_REP_TIME = 2_000
B5_OVERWRITE_TIMING_CACHE = False

if np.any(B5_VARIABLE_X_KB <= 0):
    raise ValueError("B5_VARIABLE_X_KB should contain positive half-widths")
if np.any(B5_VARIABLE_T_MIN <= B5_TIME_RANGE_START_MIN):
    raise ValueError("B5_VARIABLE_T_MIN should be larger than B5_TIME_RANGE_START_MIN so variable_t is the upper rescale endpoint")

B5_SCAN_DATASETS = build_zhao_hct116_centered_window_configs(
    center_kb=B5_CENTER_KB,
    half_widths_kb=B5_VARIABLE_X_KB,
    chrom="chr8",
    periodic=B5_PERIODIC,
    resolution=ZHAO_HCT116_ANALYSIS_RESOLUTION,
    accession=ZHAO_HCT116_GSE_ACCESSION,
    source_resolution=ZHAO_HCT116_SOURCE_RESOLUTION,
    cell_line="HCT116",
)

B5_TOPOLOGY_LABEL = "periodic" if B5_PERIODIC else "line"

display(pd.DataFrame([
    {
        "key": key,
        "topology": cfg["topology"],
        "center_kb": cfg["center_kb"],
        "variable_x_kb": cfg["variable_x_kb"],
        "coordinates": f"{cfg.get('requested_chrom', cfg['chrom'])}:{cfg['start']}-{cfg['end']}",
        "analysis_resolution_bp": cfg["resolution"],
        "bound_geometry": cfg["bound_geometries"][0],
    }
    for key, cfg in B5_SCAN_DATASETS.items()
]))

b5_timing_preprocessing = preprocess_zhao_hct116_timing_collection(
    B5_SCAN_DATASETS,
    zhao_hct116_matrix,
    overwrite=B5_OVERWRITE_TIMING_CACHE,
)
display(b5_timing_preprocessing)


In [ ]:
import contextlib
import io
import time

B5_SHOW_PROGRESS = True
B5_SAVE_FIGURE = False
B5_FIGURE_PDF = FIGURE_DIR / f"HCT116_chr8_B5_Tepsilon_abs_difference_eps{safe_filename(f'{B5_EPSILON:g}')}_{B5_TOPOLOGY_LABEL}.pdf"

b5_results = {}
b5_run_specs = [
    (float(variable_t), key, cfg)
    for variable_t in B5_VARIABLE_T_MIN
    for key, cfg in B5_SCAN_DATASETS.items()
]
b5_total_runs = len(b5_run_specs)
b5_global_start_time = time.time()

if B5_SHOW_PROGRESS:
    print(
        f"B5 scan starting | runs={b5_total_runs} | "
        f"time values={len(B5_VARIABLE_T_MIN)} | space values={len(B5_SCAN_DATASETS)} | "
        f"topology={B5_TOPOLOGY_LABEL}",
        flush=True,
    )

for b5_iteration, (variable_t, key, cfg) in enumerate(b5_run_specs, start=1):
    b5_elapsed = time.time() - b5_global_start_time
    if b5_iteration == 1:
        b5_eta_text = "estimating"
    else:
        b5_average = b5_elapsed / (b5_iteration - 1)
        b5_eta_text = hms((b5_total_runs - b5_iteration + 1) * b5_average)

    if B5_SHOW_PROGRESS:
        print(
            f"B5 global run {b5_iteration}/{b5_total_runs} | "
            f"variable_t={variable_t:g} min | "
            f"variable_x={cfg['variable_x_kb']:g} kb | "
            f"elapsed={hms(b5_elapsed)} | ETA={b5_eta_text}",
            flush=True,
        )

    b5_run_start_time = time.time()
    run_cfg = dict(cfg)
    run_cfg["variable_t_min"] = float(variable_t)
    run_cfg["key"] = f"{cfg['key']}_T{safe_filename(f'{variable_t:g}')}"
    run_cfg["label"] = f"{cfg['label']}, timing range ({B5_TIME_RANGE_START_MIN:g}, {variable_t:g}) min"
    b5_key = f"{key}_T{safe_filename(f'{variable_t:g}')}"
    b5_run_kwargs = dict(
        cfg=run_cfg,
        fork_speed_kb_min=B5_FORK_SPEED_KB_MIN,
        sim_number=B5_SIM_NUMBER,
        refine_factor=B5_REFINE_FACTOR,
        smooth_window=B5_SMOOTH_WINDOW,
        timing_range=(B5_TIME_RANGE_START_MIN, float(variable_t)),
        eps_grid=np.array([B5_EPSILON], dtype=float),
        num_t_bound=B5_NUM_T_BOUND,
        max_rep_time=B5_MAX_REP_TIME,
        timing_cache_mode="load",
    )

    with contextlib.redirect_stdout(io.StringIO()):
        b5_results[b5_key] = run_single_dataset(**b5_run_kwargs)

    if B5_SHOW_PROGRESS:
        b5_elapsed = time.time() - b5_global_start_time
        b5_average = b5_elapsed / b5_iteration
        b5_eta_text = hms((b5_total_runs - b5_iteration) * b5_average)
        print(
            f"B5 completed {b5_iteration}/{b5_total_runs} | "
            f"last={hms(time.time() - b5_run_start_time)} | "
            f"elapsed={hms(b5_elapsed)} | ETA={b5_eta_text}",
            flush=True,
        )

B5_ABS_DIFF_TABLE = completion_tepsilon_difference_table(
    b5_results,
    epsilon=B5_EPSILON,
)

B5_PLOT_TITLE = (
    f"HCT116 chr8 fixed-epsilon completion-time absolute difference, "
    f"epsilon={B5_EPSILON:g}, {B5_TOPOLOGY_LABEL}"
)
fig, ax, B5_ABS_DIFF_GRID = plot_completion_tepsilon_difference_heatmap(
    B5_ABS_DIFF_TABLE,
    title=B5_PLOT_TITLE,
)
if B5_SAVE_FIGURE:
    fig.savefig(B5_FIGURE_PDF, bbox_inches="tight")
plt.show()


# Notes on interpretation

For non-periodic chromosome profiles, the simulation is run with `perQ=False` and the comparison is made with the full-line completion bound. The local initiation mass is computed using non-wrapping intervals in the observed chromosome, not circular arcs. This keeps the analysis aligned with the line geometry.

For the HCT116 chr8 finite-window profiles, the torus completion bound is the natural theoretical object because each selected interval is being analysed as a circularised synthetic domain. The fit and stochastic simulations are therefore run with `perQ=True`. These are HCT116 Repli-seq-derived genomic-window analyses, not purified ecDNA Repli-seq measurements.

In both sections, the code converts the physical fork speed in kb/min into grid units using

$$
v_{\text{grid}} = \frac{v_{\text{kb/min}}}{dx_{\text{kb}}}.
$$

and uses this grid speed in both the stochastic simulations and the theoretical bound. The local initiation mass is computed in grid units, so fitted initiation rates are not multiplied by the input bin size. The kb conversion is used only for plotting lengths.

The expected-time summary is obtained by integrating the same survival upper bound used for the completion-time curves. It therefore matches the pointwise-uniform logic and bounds the slowest expected local replication time `max_x E[T(x)]`. For full citation details, see the README references: initiation-rate fitting follows [Berkemeier et al. (2025)](https://www.nature.com/articles/s41467-025-59991-w), and the completion-bound comparison follows Alkhaled, Berkemeier & Nik (2026).